# ST554_Final Project

Final project for ST554: Analysis of Big Data at NC State University, Spring 2026

Creator: Cole Hammett

Purpose: Predicting power consumption (Zone 3) for Tetouan City using PySpark MLlib and Structured Streaming.

Date: 30th of April, 2026

Instructor: Justin Post

# 1. Setup & Data Loading

In [1]:
# Import necessary modules

import pandas as pd
from pyspark.sql import SparkSession

## 1.1 Starting a SparkSession

`local[*]` tells Spark to run locally using all available CPU cores.

`appName` is just a label for the UI/logs.

In [2]:
# Create a SparkSession using all available local cores
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('power_zone_prediction') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 12:09:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1.2 Reading data with pandas

Two-step load, read in .csv with pandas THEN convert rather than reading directly into Spark

In [3]:
# Read power_ml_data.csv directly from the course URL into a pandas DataFrame
power_pd = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")

In [4]:
# Quick sanity checks on the pandas DataFrame
print("Shape:", power_pd.shape)
print("\nFirst few rows:")
power_pd.head()

Shape: (47174, 10)

First few rows:


,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


## 1.3 Converting to a Spark Dataframe

spark.createDataFrame() accepts a pandas DataFrame directly

In [5]:
spark_df = spark.createDataFrame(power_pd)

### 1.3.1 Verifying the schema for catching type issues before pipeline work

In particular, checked whether Hour is already a DoubleType rather than `LongType` or `IntegerType`

In [6]:
print("Spark DataFrame schema:")
spark_df.printSchema()

Spark DataFrame schema:
root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



### 1.3.2 Previewing the first few rows of the Spark DataFrame

In [7]:
spark_df.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

# 2. Transformer Pipeline

Each sub-step below becomes a stage in a `Pipeline()`. The one exception is PCA, which must be pre-fit outside the pipeline so that a fitted transformer (not an estimator) is passed in as a stage.

In [8]:
# Import all MLlib components needed for the transformation pipeline
from pyspark.ml import Pipeline
from pyspark.ml.feature import (SQLTransformer, Binarizer, OneHotEncoder,
                                 VectorAssembler, PCA)

## 2.1 Casting Hour to DoubleType

 - The schema check above showed Hour is `LongType`. 
 - `VectorAssembler` requires `DoubleType`.
 - SQLTransformer rewrites the whole table in-place using `__THIS__` as the table alias.
 - Casted Hour to DoubleType under a new name to avoid a duplicate-column conflict.
 - `SELECT *` keeps the original `LongType` Hour; the cast result goes into `Hour_d`.
 
Note: `sql_cast` uses `SELECT *, CAST(...)` rather than listing every column which keeps all original columns intact so downstream stages can still reference them.

In [9]:
sql_cast = SQLTransformer(
    statement="SELECT *, CAST(Hour AS DOUBLE) AS Hour_d FROM __THIS__"
)

## 2.2 Binarizing `Hour`

- Binarized Hour: 0 if Hour <= 6.5 (night), 1 if Hour > 6.5 (day).
- inputCol read the freshly-cast DoubleType Hour produced by sql_cast above.

In [10]:
binarizer = Binarizer(
    threshold=6.5,
    inputCol="Hour_d",
    outputCol="Hour_bin"
)

## 2.3 One-Hot Encoding `Month`

Month is already numeric (1–12), so no StringIndexer is needed. OHE will produce a sparse vector of length 11 (one category dropped by default).

In [11]:
ohe = OneHotEncoder(
    inputCol="Month",
    outputCol="Month_ohe"
)

## 2.4 Pre-fitting PCA on weather columns

The trickiest part. `pca_estimator.fit()` returns a `PCAModel` (transformer); that's what goes into the pipeline, not `pca_estimator` itself.

### 2.4.1 Assembling the five weather columns into a single vector column

`weather_assembler` appears as a pipeline stage so it runs on streaming data later. Cannot just rely on the `weather_vec_df` created during the pre-fit.

In [12]:
weather_assembler = VectorAssembler(
    inputCols=["Temperature", "Humidity", "Wind_Speed",
               "General_Diffuse_Flows", "Diffuse_Flows"],
    outputCol="weather_vec"
)

### 2.4.2 Fitting the PCA estimator on the assembled weather vectors

Fitted here (outside the pipeline) so that the *fitted* PCA model(a transformer) is passed into the pipeline rather than the unfitted estimator.

In [13]:
pca_estimator = PCA(k=2, inputCol="weather_vec", outputCol="pca_features")

weather_vec_df = weather_assembler.transform(spark_df)
pca_model = pca_estimator.fit(weather_vec_df)

print("PCA explained variance ratios:", pca_model.explainedVariance)

PCA explained variance ratios: [0.883862615096907,0.11355803317669583]


## 2.5 Renaming response as `label`

Renamed Power_Zone_3 to label as required by MLlib's LinearRegression. SELECT * preserved all existing columns while adding the alias.

In [14]:
sql_label = SQLTransformer(
    statement="SELECT *, Power_Zone_3 AS label FROM __THIS__"
)

## 2.6 Combining all predictors into a single 'features' vector

- 2 PCA components from the weather columns
- binary Hour flag (night vs. day)
- Power_Zone_1 and Power_Zone_2 (known zone readings)
- OHE Month indicators (sparse vector)

In [15]:
final_assembler = VectorAssembler(
    inputCols=["pca_features", "Hour_bin",
               "Power_Zone_1", "Power_Zone_2", "Month_ohe"],
    outputCol="features"
)

## 2.7 Assembling the pipeline

Built the full transformation pipeline using all stages defined above. Note: pca_model is the fitted PCA transformer, not the raw PCA estimator.

In [16]:
pipeline = Pipeline(stages=[
    sql_cast,          # 2a — cast Hour to Double
    binarizer,         # 2b — binarize Hour into night/day flag
    ohe,               # 2c — one-hot encode Month
    weather_assembler, # 2d (part 1) — bundle weather cols into a vector
    pca_model,         # 2d (part 2) — apply pre-fitted PCA (2 components)
    sql_label,         # 2e — alias Power_Zone_3 as label
    final_assembler    # 2f — assemble all predictors into features vector
])

## 2.8 Verifying the pipeline

Transformed the data and confirmed features + label columns exist

In [17]:
pipeline_check = pipeline.fit(spark_df).transform(spark_df)
pipeline_check.select("label", "features").show(5, truncate=True)

+-----------+--------------------+
|      label|            features|
+-----------+--------------------+
|20240.96386|(17,[0,1,3,4,6],[...|
|20131.08434|(17,[0,1,3,4,6],[...|
|19668.43373|(17,[0,1,3,4,6],[...|
|18899.27711|(17,[0,1,3,4,6],[...|
|18442.40964|(17,[0,1,3,4,6],[...|
+-----------+--------------------+
only showing top 5 rows


# 3. Cross-Validated Elastic Net

Built a `LinearRegression` estimator, defined the 11×11 parameter grid(121 combinations), wrapped the pipeline inside a `CrossValidator`, and fitted on the full dataset.

Note: 121 combinations × 5 folds = 605 model fits

In [18]:
# Imports for model fitting, hyperparameter search, and evaluation
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

## 3.1 Building the parameter grid

### 3.1.1 Instantiating a LinearRegression estimator 

Elastic net is controlled via regParam + elasticNetParam

In [19]:
lr = LinearRegression()

### 3.1.2 Building an 11x11 grid 

All combinations of regParam and elasticNetParam = 121 total

In [20]:
param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam,        [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1])
    .build()
)

print(f"Total parameter combinations: {len(param_grid)}")  # should be 121

Total parameter combinations: 121


## 3.2 Configuring the `CrossValidator`

- Wrapped the full transformation pipeline + LR model in a `CrossValidator` 
- `estimator` = pipeline (all feature engineering + LR in one object) 
- `numFolds=5` with RMSE as the selection criterion; no train/test split per assignment instructions

In [21]:
crossval = CrossValidator(
    estimator=Pipeline(stages=pipeline.getStages() + [lr]),
    estimatorParamMaps=param_grid,
    evaluator=RegressionEvaluator(metricName="rmse"),
    numFolds=5
)

## 3.3 Fitting the `CrossValidator`

On the full dataset (605 model fits total)

In [22]:
cv_model = crossval.fit(spark_df)
print("Cross-validation complete.")

26/04/30 12:09:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/30 12:09:55 WARN Instrumentation: [d5b64fd5] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 12:09:56 WARN Instrumentation: [d5b64fd5] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/30 12:09:58 WARN Instrumentation: [9eec1ae6] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 12:09:58 WARN Instrumentation: [9eec1ae6] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/30 12:09:59 WARN Instrumentation: [205d7489] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 12:09:59 WARN Instrumentation: [205d7489] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

Cross-validation complete.


# 4. Report Results

Extract the best tuning parameters, CV RMSE, and training RMSE from the fitted cross-validator.

## 4.1 Finding optimal tuning parameters

`cv_model.bestModel` is the `PipelineModel` that won the grid search. Its `.stages[-1]` is the fitted `LinearRegression`, and `.getRegParam()` / `.getElasticNetParam()` pull the exact values Spark chose.

### 4.1.1 Pulling the best LinearRegression model 

Located the best model out of the winning pipeline. Pulling the last stage (the fitted LinearRegression)

In [23]:
best_lr = cv_model.bestModel.stages[-1]

### 4.1.2 Extracting the optimal elastic net tuning parameters

In [24]:
best_reg_param        = best_lr.getRegParam()
best_elastic_net_param = best_lr.getElasticNetParam()

print(f"Optimal regParam:        {best_reg_param}")
print(f"Optimal elasticNetParam: {best_elastic_net_param}")

Optimal regParam:        0.05
Optimal elasticNetParam: 0.1


## 4.2 Getting best CV RMSE

`cv_model.avgMetrics` stores a list of average fold-RMSE values 
- one per grid combination and 
- in the same order as the `paramGrid` 

The minimum, given by `min()`, is the CV error for the chosen best model (the winning parameter set).

In [25]:
cv_rmse = min(cv_model.avgMetrics)

print(f"5-fold CV RMSE: {cv_rmse:.4f}")

5-fold CV RMSE: 2147.9955


## 4.3 Getting Training RMSE

### 4.3.1 Using the fitted cv_model on the full training set

Called `.transform(spark_df)` on the cross-validator model to run the winning pipeline (all transformations and best LR) over the original data and produce a `prediction` column 

In [27]:
train_predictions = cv_model.transform(spark_df)

### 4.3.2 Evaluating predictions against the true labels using RMSE

Computed the RMSE of `prediction` against `label` using the `RegressionEvaluator`.

In [28]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol = "label",
    predictionCol = "prediction",
    metricName = "rmse"
)

train_rmse = evaluator.evaluate(train_predictions)
print(f"Training Set RMSE: {train_rmse:.4f}")

Training Set RMSE: 2147.0973


Note: The Training Set RMSE will always be lower than the CV RMSE since the model has seen this data. 